In [ ]:
"""
Autor: Jairo de Jesús Blanco Pérez
Codigo: <tu código>
Materia: Inteligencia Artificial 2026-2
Tema: Fase exploratoria: máscaras con SAM 2, C: reentrenamiento de la U-Net v2 y comparación visual
"""

# C. Reentrenamiento de la U-Net v2 y comparación visual con la v1

Este cuaderno es la parte C de la fase exploratoria con SAM 2. Reentrenamos **la misma U-Net desde cero** cambiando **solo las etiquetas** (pseudo-máscaras de SAM 2 en lugar de GrabCut): misma arquitectura, mismos hiperparámetros, mismo aumento y misma semilla. Así cualquier diferencia se puede atribuir a las etiquetas.

Reglas que respetamos:

- No se modifica ningún artefacto de la v1 (solo lectura).
- Contratos de preprocesamiento: imagen RGB $256\times256$ con `INTER_AREA` y dividida entre $255$; máscaras reducidas con `INTER_NEAREST` (valores $\{0,255\}$). La CNN no interviene.
- El mIoU de la v2 se mide **contra pseudo-máscaras de SAM 2**, así que **no es comparable** con el $0.8334$ de la v1 (medido contra GrabCut). La comparación v1 contra v2 es **visual, a ojo**; no hay anotación manual ni IoU contra anotaciones.

Constantes que debemos fijar nosotros: `REGLA_VENTAJA_AZAR` (inicio de la sección 6, **antes** de mirar las cuadrículas), `REENTRENAR` (sección 4) y `CONFIRMO_TEST` (sección 8).

## 0. Configuración

Ahora vamos a importar todas las librerías en una sola celda.

In [ ]:
import os  # rutas y archivos
import json  # lectura y escritura de JSON
import time  # cronómetro por época
import random  # semilla de Python
import hashlib  # MD5 de splits.csv y del modelo
import zipfile  # lectura de masks_sam2.zip y masks.zip
import datetime  # fecha de los registros
import numpy as np  # arreglos numéricos
import pandas as pd  # tablas
import cv2  # lectura, redimensionado y contornos
import matplotlib.pyplot as plt  # figuras
import tensorflow as tf  # TensorFlow
from tensorflow import keras  # Keras
from tensorflow.keras import layers  # capas de la U-Net
from google.colab import drive  # montaje de Drive

Ahora vamos a definir las constantes y las rutas. `ETIQUETAS = "sam2"` es la **única** diferencia respecto a la v1: indica qué máscaras se usan como objetivo y sirve de prefijo en los nombres de las salidas. Todas las rutas son relativas a `ROOT`.

In [ ]:
SEED = 42  # semilla global (random, numpy, tf)
ETIQUETAS = "sam2"  # origen de las etiquetas de la v2: pseudo-máscaras de SAM 2 (v1 usó GrabCut); prefijo de las salidas
assert ETIQUETAS == "sam2"  # este cuaderno solo está escrito para SAM 2
ROOT = '/content/drive/MyDrive/sylfish'  # raíz del proyecto en Drive
MD5_SPLITS = '5b174a510317656566c44916ff8da4a7'  # MD5 esperado de splits.csv
EPS = 1e-6  # épsilon de Dice e IoU
IMG = 256  # lado de entrada de la U-Net
BATCH = 16  # tamaño de lote
EPOCAS_MAX = 40  # máximo de épocas
PARAMS_ESPERADOS = 1946993  # parámetros totales de la U-Net (igual que la v1)
TAUS = np.round(np.arange(0.10, 0.91, 0.05), 2)  # umbrales a barrer en val (igual que el 03)
SPLITS = ('train', 'val', 'test')  # particiones
ESTRATOS = ('aleatoria', 'peor', 'descartada')  # estratos de muestra_val.csv
N_ESTRATO = {'aleatoria': 15, 'peor': 8, 'descartada': 7}  # tamaño esperado de cada estrato

P_SPLITS = f'{ROOT}/data/splits/splits.csv'  # partición oficial
P_ZIP_SAM2 = f'{ROOT}/sam2/masks_sam2.zip'  # máscaras SAM 2 (cuaderno B)
P_ZIP_GRABCUT = f'{ROOT}/data/masks.zip'  # máscaras GrabCut (solo para mostrar)
P_QC = f'{ROOT}/sam2/sam2_mask_qc.csv'  # control de calidad de las máscaras SAM 2 (cuaderno B)
P_MUESTRA = f'{ROOT}/sam2/muestra_val.csv'  # 30 imágenes de val (cuaderno A)
ZIP_IMG = f'{ROOT}/data/raw/SylFishBD_IMAGE.zip'  # imágenes originales (solo existe el zip)
DIR_IMG_LOCAL = '/content/sylfish_images'  # carpeta local de extracción
DIR_PROC = f'{ROOT}/data/processed'  # cachés unet_256_*.npz de entrada (solo lectura)
DIR_CACHE_SAM2 = f'{ROOT}/sam2/cache'  # cachés nuevas de máscaras SAM 2 (salida)
DIR_MODELS = f'{ROOT}/models'  # modelos
DIR_TABLAS = f'{ROOT}/reports/tables'  # tablas y json
DIR_FIGS = f'{ROOT}/reports/figures'  # figuras
P_V1 = f'{DIR_MODELS}/unet_v1.keras'  # v1 (solo lectura)
P_V1_TAU = f'{DIR_MODELS}/unet_v1_umbral.json'  # umbral de la v1 (solo lectura)
P_V2 = f'{DIR_MODELS}/unet_v2.keras'  # v2 (salida)
P_V2_TAU = f'{DIR_MODELS}/unet_v2_umbral.json'  # umbral de la v2 (salida)
P_HIST = f'{DIR_TABLAS}/{ETIQUETAS}_unet_v2_historial.csv'  # historial (CSVLogger)
P_MARCA = f'{DIR_TABLAS}/{ETIQUETAS}_unet_v2_entrenamiento_ok.json'  # marca de entrenamiento completo
P_SELECCION = f'{DIR_TABLAS}/{ETIQUETAS}_seleccion.json'  # decisión v1/v2
P_REG_TEST = f'{ROOT}/reports/{ETIQUETAS}_unet_v2_test_registro.json'  # registro de uso único del test
P_RES_TEST = f'{DIR_TABLAS}/{ETIQUETAS}_unet_v2_test_resumen.json'  # resumen del test


def p_cache_unet(split):  # caché de la U-Net v1 (imágenes, máscaras GrabCut, filepath)
    return f'{DIR_PROC}/unet_256_{split}.npz'


def p_cache_sam2(split):  # caché de máscaras SAM 2 (salida de la sección 1)
    return f'{DIR_CACHE_SAM2}/{ETIQUETAS}_unet_256_masks_{split}.npz'

Ahora vamos a montar Drive, crear las carpetas de salida y comprobar el MD5 de `splits.csv`; si no coincide, algo cambió en la partición y no seguimos.

In [ ]:
drive.mount('/content/drive')  # monta Drive
assert os.path.isdir(ROOT), f'No existe {ROOT}'  # la raíz debe existir
for d in (DIR_CACHE_SAM2, DIR_MODELS, DIR_TABLAS, DIR_FIGS, DIR_IMG_LOCAL):  # carpetas de salida
    os.makedirs(d, exist_ok=True)  # crea si faltan
with open(P_SPLITS, 'rb') as f:  # lee splits.csv en binario
    md5 = hashlib.md5(f.read()).hexdigest()  # MD5 del archivo
print('MD5 splits.csv:', md5)
assert md5 == MD5_SPLITS, 'splits.csv cambió: la partición ya no es la oficial'  # control de la partición

Ahora vamos a extraer las imágenes originales a disco local. En Drive solo existe el zip `data/raw/SylFishBD_IMAGE.zip` (no hay carpeta `images/`). Indexamos por el nombre de archivo sin extensión (`stem`, único en el dataset), así no dependemos de cómo esté organizado el zip por dentro.

In [ ]:
marca = f'{DIR_IMG_LOCAL}/.listo'                                  # se escribe al terminar la extracción
if not os.path.exists(marca):                                       # extrae una sola vez por sesión (~420 MB)
    with zipfile.ZipFile(ZIP_IMG) as zf:
        zf.extractall(DIR_IMG_LOCAL)
    open(marca, 'w').close()
IMG_INDEX = {os.path.splitext(f)[0]: os.path.join(r, f)             # stem -> ruta local
             for r, _, fs in os.walk(DIR_IMG_LOCAL) for f in fs if f.lower().endswith(('.jpg', '.jpeg', '.png'))}

def ruta_img(fp):                                                   # ruta local de una imagen a partir de su filepath
    return IMG_INDEX[os.path.splitext(os.path.basename(fp))[0]]

print(len(IMG_INDEX), 'imágenes indexadas (esperado: 9075)')

Ahora vamos a revisar la GPU y a fijar las semillas (Python, NumPy y TensorFlow con el mismo valor $42$), con crecimiento de memoria activado.

In [ ]:
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)  # versiones
gpus = tf.config.list_physical_devices('GPU')  # GPUs visibles
print('GPUs:', gpus)
for g in gpus:  # una por una
    tf.config.experimental.set_memory_growth(g, True)  # no reserva toda la memoria
if not gpus:  # sin GPU el entrenamiento sería inviable
    print('AVISO: no hay GPU. En Colab: Entorno de ejecución > Cambiar tipo > GPU T4.')
random.seed(SEED)  # semilla de Python
np.random.seed(SEED)  # semilla de NumPy
tf.random.set_seed(SEED)  # semilla de TensorFlow
keras.utils.set_random_seed(SEED)  # fija las tres a la vez

Ahora vamos a definir utilidades pequeñas: escritura **atómica** (archivo temporal y `os.replace`) de JSON y figuras, y la función que obtiene el *stem* (nombre sin carpeta ni extensión) con el que emparejamos los archivos de los zips.

In [ ]:
def escribir_json(obj, ruta):  # escritura atómica de JSON
    tmp = ruta + '.tmp'  # archivo temporal junto al destino
    with open(tmp, 'w', encoding='utf-8') as f:  # escribe el temporal
        json.dump(obj, f, ensure_ascii=False, indent=2)  # JSON legible
    os.replace(tmp, ruta)  # reemplazo atómico


def guardar_figura(fig, ruta):  # escritura atómica de figura PNG
    tmp = ruta + '.tmp.png'  # temporal con extensión válida
    fig.savefig(tmp, dpi=130, bbox_inches='tight')  # guarda el temporal
    os.replace(tmp, ruta)  # reemplazo atómico


def stem(fp):  # 'clase/boal_001.jpg' -> 'boal_001'
    return os.path.splitext(os.path.basename(fp))[0]  # nombre sin carpeta ni extensión


def md5_archivo(ruta):  # MD5 de un archivo grande por bloques
    h = hashlib.md5()  # acumulador
    with open(ruta, 'rb') as f:  # lectura binaria
        for bloque in iter(lambda: f.read(1 << 20), b''):  # bloques de 1 MiB
            h.update(bloque)  # actualiza
    return h.hexdigest()  # resumen hexadecimal

## 1. Caché de máscaras SAM 2 a $256\times256$

El cuaderno B dejó `masks_sam2.zip` (PNG en $\{0,255\}$ de $500\times500$) y `sam2_mask_qc.csv`. Aquí producimos, para cada partición, `sam2_unet_256_masks_{split}.npz` con las claves `M`, `filepath` y `sam2_qc_ok`, **en el mismo orden** que `unet_256_{split}.npz`. Cruzamos siempre por `filepath` (y por *stem* para el zip), nunca por posición.

Ahora vamos a leer `splits.csv`, el control de calidad y el índice del zip (diccionario *stem* $\to$ ruta interna construido desde `namelist()`).

In [ ]:
splits = pd.read_csv(P_SPLITS)  # partición oficial
assert {'filepath', 'clase', 'label', 'group_id', 'split'} <= set(splits.columns)  # columnas del contrato
assert splits['filepath'].is_unique  # una fila por imagen
splits['stem'] = splits['filepath'].map(stem)  # stem de cada imagen
assert splits['stem'].is_unique, 'hay stems repetidos: el emparejamiento por stem sería ambiguo'  # requisito del cruce
cuentas = splits['split'].value_counts().to_dict()  # tamaños por partición
print(cuentas)
assert cuentas == {'train': 6246, 'val': 1359, 'test': 1370}  # tamaños esperados

qc = pd.read_csv(P_QC)  # control de calidad de SAM 2
assert qc['filepath'].is_unique  # una fila por imagen
qc['ok'] = qc['sam2_qc_ok'].astype(str).str.strip().str.lower().isin(['true', '1', '1.0'])  # a booleano robusto
assert not (set(splits['filepath']) - set(qc['filepath'])), 'el QC no cubre todas las imágenes'  # cobertura total
QC_OK = dict(zip(qc['filepath'], qc['ok']))  # filepath -> sam2_qc_ok
print(splits.assign(ok=splits['filepath'].map(QC_OK)).groupby('split')['ok'].agg(['sum', 'count']))  # ok por split

with zipfile.ZipFile(P_ZIP_SAM2) as zf:  # abre el zip de SAM 2
    nombres = [n for n in zf.namelist()  # entradas del zip
               if n.lower().endswith('.png') and '__MACOSX' not in n and not os.path.basename(n).startswith('.')]  # solo PNG reales
ZIP_SAM2 = {stem(n): n for n in nombres}  # stem -> ruta interna
assert len(ZIP_SAM2) == len(nombres), 'stems repetidos dentro de masks_sam2.zip'  # sin colisiones
print('PNG en masks_sam2.zip:', len(ZIP_SAM2))

Ahora vamos a definir la construcción **idempotente**: si la caché existe y es consistente (mismo orden de `filepath`, mismos `sam2_qc_ok`, forma, tipo y valores $\{0,255\}$) se conserva; si no, se reconstruye y se escribe de forma atómica. Las máscaras se reducen con `INTER_NEAREST`, así que siguen siendo binarias.

In [ ]:
def cache_consistente(ruta, fps, ok):  # ¿la caché existente es válida?
    if not os.path.exists(ruta):  # no existe
        return False
    try:
        c = np.load(ruta, allow_pickle=True)  # abre el npz
        if set(c.files) != {'M', 'filepath', 'sam2_qc_ok'}:  # claves del contrato
            return False
        if not np.array_equal(c['filepath'].astype(str), fps):  # mismo orden y mismos filepath
            return False
        if not np.array_equal(c['sam2_qc_ok'].astype(bool), ok):  # mismo QC
            return False
        M = c['M']  # máscaras
        return bool(M.dtype == np.uint8 and M.shape == (len(fps), IMG, IMG) and np.all((M == 0) | (M == 255)))  # forma y valores
    except Exception as e:  # archivo corrupto o incompleto
        print('  caché ilegible:', e)
        return False


def construir_cache_sam2(split):  # construye (o carga) la caché de un split
    ruta = p_cache_sam2(split)  # destino
    ref = np.load(p_cache_unet(split), allow_pickle=True)  # caché de la U-Net v1 (referencia de orden)
    fps = ref['filepath'].astype(str)  # orden de referencia
    ok = np.array([QC_OK[f] for f in fps], dtype=bool)  # sam2_qc_ok por filepath
    if cache_consistente(ruta, fps, ok):  # idempotencia
        print(f'{split}: caché existente y consistente, se conserva')
        return
    M = np.zeros((len(fps), IMG, IMG), np.uint8)  # máscaras 256x256
    faltan = 0  # máscaras ausentes en el zip
    with zipfile.ZipFile(P_ZIP_SAM2) as zf:  # abre el zip
        for i, fp in enumerate(fps):  # por imagen, en el orden de referencia
            s = stem(fp)  # stem de la imagen
            if s not in ZIP_SAM2:  # sin PNG
                assert not ok[i], f'falta la máscara de {fp} con sam2_qc_ok=True'  # solo se tolera si el QC la descartó
                faltan += 1
                continue
            m = cv2.imdecode(np.frombuffer(zf.read(ZIP_SAM2[s]), np.uint8), cv2.IMREAD_GRAYSCALE)  # decodifica el PNG
            assert m is not None and m.shape == (500, 500), f'PNG inválido: {s}'  # contrato 500x500
            assert np.all((m == 0) | (m == 255)), f'valores fuera de {{0,255}}: {s}'  # binaria
            m = cv2.resize(m, (IMG, IMG), interpolation=cv2.INTER_NEAREST)  # reducción sin interpolar valores
            if ok[i]:
                assert m.any(), f'máscara vacía con sam2_qc_ok=True: {s}'  # una máscara aprobada no puede estar vacía
            M[i] = m  # guarda en su posición (el orden es el de la caché de referencia)
    tmp = ruta + '.tmp.npz'  # temporal (debe terminar en .npz)
    np.savez_compressed(tmp, M=M, filepath=fps, sam2_qc_ok=ok)  # escribe el temporal
    os.replace(tmp, ruta)  # reemplazo atómico
    assert cache_consistente(ruta, fps, ok)  # releer y verificar
    print(f'{split}: {len(fps)} máscaras escritas ({int(ok.sum())} con sam2_qc_ok, {faltan} sin PNG)')

Ahora vamos a construir las tres cachés (train, val y test).

In [ ]:
for split in SPLITS:  # cada partición
    t0 = time.time()  # cronómetro
    construir_cache_sam2(split)  # construye o carga
    print(f'   {time.time() - t0:.1f} s')

Ahora vamos a **verificar desde disco** que cada caché nueva coincide con la de la U-Net en orden y en valores de `filepath`, que contiene exactamente las imágenes del split en `splits.csv`, y que `M` es `uint8` con valores solo en $\{0,255\}$.

In [ ]:
for split in SPLITS:  # cada partición
    ref = np.load(p_cache_unet(split), allow_pickle=True)  # caché U-Net v1
    c = np.load(p_cache_sam2(split), allow_pickle=True)  # caché SAM 2
    fr = ref['filepath'].astype(str)  # filepath de referencia
    fc = c['filepath'].astype(str)  # filepath de la caché SAM 2
    assert np.array_equal(fr, fc), f'{split}: el orden de filepath no coincide'  # mismo orden
    esperados = set(splits.loc[splits['split'] == split, 'filepath'])  # imágenes del split según splits.csv
    assert len(fr) == len(esperados) and set(fr) == esperados, f'{split}: las imágenes no son las del split'  # mismas imágenes
    assert ref['y'].shape[0] == len(fr)  # etiquetas de clase alineadas
    M, ok = c['M'], c['sam2_qc_ok']  # máscaras y QC
    assert M.dtype == np.uint8 and M.shape == (len(fr), IMG, IMG)  # tipo y forma
    assert np.all((M == 0) | (M == 255))  # solo {0,255}
    assert ok.dtype == bool and len(ok) == len(fr)  # QC booleano
    frac = (M[ok] > 0).mean(axis=(1, 2))  # fracción de área de las máscaras aprobadas
    print(f'{split}: N={len(fr)} | sam2_qc_ok={int(ok.sum())} | área mediana de la máscara={np.median(frac):.3f}')
print('Verificación de caché: correcta')

Ahora vamos a mirar una cuadrícula de $8$ imágenes por partición (elegidas al azar con la semilla $42$) con el contorno de la máscara SAM 2 sobre la imagen de la caché de la U-Net. Si el contorno no rodea al pez, hay un desalineamiento y debemos parar.

In [ ]:
rng = np.random.default_rng(SEED)  # generador reproducible
for split in SPLITS:  # una cuadrícula por partición
    ref = np.load(p_cache_unet(split), allow_pickle=True)  # caché U-Net
    c = np.load(p_cache_sam2(split), allow_pickle=True)  # caché SAM 2
    X, M, fps, ok = ref['X'], c['M'], c['filepath'].astype(str), c['sam2_qc_ok']  # datos
    idx = np.sort(rng.choice(len(fps), 8, replace=False))  # 8 índices al azar
    fig, ejes = plt.subplots(2, 4, figsize=(16, 8))  # cuadrícula 2x4
    for ax, i in zip(ejes.ravel(), idx):  # cada imagen
        img = X[i].copy()  # copia contigua para dibujar
        cont, _ = cv2.findContours((M[i] > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # contornos
        cv2.drawContours(img, cont, -1, (255, 0, 0), 1)  # contorno rojo
        ax.imshow(img)  # imagen RGB
        ax.set_title(f'{stem(fps[i])} | qc_ok={bool(ok[i])}', fontsize=9)  # identificación
        ax.axis('off')
    fig.suptitle(f'Verificación de caché SAM 2 - {split} (contorno rojo = máscara SAM 2)')  # título
    guardar_figura(fig, f'{DIR_FIGS}/{ETIQUETAS}_cache_verificacion_{split}.png')  # guarda
    plt.show()
    del X, M, ref, c  # libera memoria

## 2. Datos con `tf.data`

La selección por `sam2_qc_ok == True` se aplica **al cargar** (train y val; las cachés guardan todas las imágenes). En train hacemos una permutación única con semilla $42$ antes de crear el dataset, porque las cachés están ordenadas por clase; además `shuffle(2048)` en cada época. El test no se carga aquí (sección 8).

Ahora vamos a definir la carga y a imprimir cuántas imágenes quedan por split.

In [ ]:
def cargar_split(split, permutar=False):  # carga un split ya filtrado por sam2_qc_ok
    ref = np.load(p_cache_unet(split), allow_pickle=True)  # caché U-Net (imágenes)
    c = np.load(p_cache_sam2(split), allow_pickle=True)  # caché SAM 2 (máscaras)
    fx = ref['filepath'].astype(str)  # filepath de las imágenes
    fm = c['filepath'].astype(str)  # filepath de las máscaras
    assert np.array_equal(fx, fm), f'{split}: imágenes y máscaras desalineadas'  # cruce por filepath
    ok = c['sam2_qc_ok'].astype(bool)  # selección de calidad
    idx = np.flatnonzero(ok)  # índices aprobados
    if permutar:  # permutación única con semilla 42
        idx = idx[np.random.RandomState(SEED).permutation(len(idx))]
    Xtodo = ref['X']  # lee todas las imágenes
    X = Xtodo[idx]  # copia filtrada (y permutada)
    del Xtodo  # libera la lectura completa
    M = c['M'][idx][..., np.newaxis]  # máscaras con canal (N,256,256,1), uint8 {0,255}
    print(f'{split}: {len(fx)} imágenes en la caché -> {len(idx)} con sam2_qc_ok=True')
    return X, M, fx[idx]


X_tr, M_tr, fp_tr = cargar_split('train', permutar=True)  # train filtrado y permutado
X_va, M_va, fp_va = cargar_split('val')  # val filtrado, en el orden de la caché
print(X_tr.shape, X_tr.dtype, M_tr.shape, M_tr.dtype)

Ahora vamos a definir el pipeline. La entrada pasa a `float32` dividida entre $255$ y la máscara a $\{0,1\}$ con forma $(256,256,1)$. El aumento (solo train) apila imagen y máscara en un tensor de 4 canales para aplicar **exactamente** la misma transformación a ambas: volteos horizontal y vertical y rotación de $k\cdot 90^\circ$ con $k\in\{0,1,2,3\}$; el brillo ($\pm0.15$) y el contraste ($[0.8,1.2]$) solo tocan la imagen, que luego se recorta a $[0,1]$.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE  # paralelismo automático


def preparar(x, m):  # uint8 -> float32 y máscara a {0,1}
    x = tf.cast(x, tf.float32) / 255.0  # imagen en [0,1]
    m = tf.cast(m, tf.float32) / 255.0  # máscara en {0,1}
    return x, m


def aumentar(x, m):  # aumento geométrico conjunto y fotométrico solo en la imagen
    z = tf.concat([x, m], axis=-1)  # tensor (256,256,4): RGB + máscara
    z = tf.image.random_flip_left_right(z)  # volteo horizontal conjunto
    z = tf.image.random_flip_up_down(z)  # volteo vertical conjunto
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)  # k aleatorio en 0..3
    z = tf.image.rot90(z, k)  # rotación de k*90 grados conjunta
    z = tf.ensure_shape(z, [IMG, IMG, 4])  # forma estática
    x, m = z[..., :3], z[..., 3:]  # separa imagen y máscara
    x = tf.image.random_brightness(x, 0.15)  # brillo +-0.15 (solo imagen)
    x = tf.image.random_contrast(x, 0.8, 1.2)  # contraste en [0.8,1.2] (solo imagen)
    x = tf.clip_by_value(x, 0.0, 1.0)  # recorta a [0,1]
    return x, m


def crear_dataset(X, M, entrenamiento):  # arma el pipeline
    ds = tf.data.Dataset.from_tensor_slices((X, M))  # (uint8, uint8)
    if entrenamiento:  # solo train
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)  # mezcla por época
    ds = ds.map(preparar, num_parallel_calls=AUTOTUNE)  # normaliza
    if entrenamiento:  # solo train
        ds = ds.map(aumentar, num_parallel_calls=AUTOTUNE)  # aumento
    return ds.batch(BATCH).prefetch(AUTOTUNE)  # lotes de 16

Ahora vamos a crear los datasets de train (con aumento) y val (sin aumento) y a comprobar rangos de un lote.

In [ ]:
ds_tr = crear_dataset(X_tr, M_tr, True)  # train con aumento
ds_va = crear_dataset(X_va, M_va, False)  # val sin aumento y sin mezcla (orden = X_va)
xb, mb = next(iter(ds_tr))  # un lote de train
print('lote train:', xb.shape, mb.shape, xb.dtype, mb.dtype)
print('imagen en [0,1]:', float(tf.reduce_min(xb)), float(tf.reduce_max(xb)))
print('valores de la máscara:', np.unique(mb.numpy()))
assert 0.0 <= float(tf.reduce_min(xb)) and float(tf.reduce_max(xb)) <= 1.0  # rango de la imagen
assert set(np.unique(mb.numpy())) <= {0.0, 1.0}  # máscara binaria (el aumento no la interpola)
print('batches por época: train', len(ds_tr), '| val', len(ds_va))

Ahora vamos a comprobar a ojo que el aumento mueve imagen y máscara **a la vez**: el contorno de la máscara debe seguir rodeando al pez en el lote aumentado.

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(16, 4))  # 4 ejemplos del lote
for ax, k in zip(ejes, range(4)):  # cada ejemplo
    img = (xb[k].numpy() * 255).astype(np.uint8)  # imagen aumentada en uint8
    cont, _ = cv2.findContours((mb[k, ..., 0].numpy() > 0.5).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # contornos
    cv2.drawContours(img, cont, -1, (255, 0, 0), 1)  # contorno rojo
    ax.imshow(img)  # muestra
    ax.axis('off')
fig.suptitle('Lote aumentado: el contorno rojo debe seguir rodeando al pez')  # título
plt.show()

## 3. Modelo, pérdida y métrica

Ahora vamos a montar la U-Net capa por capa, **idéntica a la del 03** (la v1): la entrada es una imagen de $256 \times 256 \times 3$ y la salida es un mapa de $256 \times 256 \times 1$ con la probabilidad de que cada píxel sea pez.

Encoder. Cuatro niveles (filtros $16, 32, 64, 128$); en cada uno dos convoluciones $3 \times 3$ con BatchNormalization y ReLU, guardamos la salida para la skip connection y reducimos con MaxPool $2 \times 2$.

Bottleneck. Dos convoluciones $3 \times 3$ con $256$ filtros a $16 \times 16$.

Decoder. Cuatro niveles simétricos: una convolución transpuesta $2 \times 2$ con `strides=2`, concatenación con la skip del mismo nivel y dos convoluciones $3 \times 3$ con BatchNormalization y ReLU.

Salida. Una convolución $1 \times 1$ con 1 filtro y activación sigmoide:

$$\hat{y}_p = \sigma(z_p) = \frac{1}{1 + e^{-z_p}} \in (0, 1)$$

Un píxel $p$ se clasifica como pez si $\hat{y}_p > \tau$.

In [ ]:
# fijamos una semilla
tf.random.set_seed(SEED)

# fijamos tamaño de las imagenes
img_size = (256,256,3)

entrada = layers.Input(shape=img_size)

# ahora pasamos a la primera parte de la arquitectura, el encoder, que reduce las dimensiones y extrae caracteristicas semanticas de aprendizaje

# Encoder - nivel 1
e1 = layers.Conv2D(16, (3, 3), padding='same')(entrada)
e1 = layers.BatchNormalization()(e1)
e1 = layers.Activation('relu')(e1)
e1 = layers.Conv2D(16, (3, 3), padding='same')(e1)
e1 = layers.BatchNormalization()(e1)
e1 = layers.Activation('relu')(e1)
skip_connection1 = e1 # guardamos la salida del nivel 1 en una skip connection que luego vamos a concatenar
e1 = layers.MaxPooling2D((2, 2))(e1)

# Encoder - nivel 2

e2 = layers.Conv2D(32, (3, 3), padding='same')(e1)
e2 = layers.BatchNormalization()(e2)
e2 = layers.Activation('relu')(e2)
e2 = layers.Conv2D(32, (3, 3), padding='same')(e2)
e2 = layers.BatchNormalization()(e2)
e2 = layers.Activation('relu')(e2)
skip_connection2 = e2
e2 = layers.MaxPooling2D((2, 2))(e2)

# Encoder - nivel 3

e3 = layers.Conv2D(64, (3, 3), padding='same')(e2)
e3 = layers.BatchNormalization()(e3)
e3 = layers.Activation('relu')(e3)
e3 = layers.Conv2D(64, (3, 3), padding='same')(e3)
e3 = layers.BatchNormalization()(e3)
e3 = layers.Activation('relu')(e3)
skip_connection3 = e3
e3 = layers.MaxPooling2D((2, 2))(e3)

# Encoder - nivel 4

e4 = layers.Conv2D(128, (3, 3), padding='same')(e3)
e4 = layers.BatchNormalization()(e4)
e4 = layers.Activation('relu')(e4)
e4 = layers.Conv2D(128, (3, 3), padding='same')(e4)
e4 = layers.BatchNormalization()(e4)
e4 = layers.Activation('relu')(e4)
skip_connection4 = e4
e4 = layers.MaxPooling2D((2, 2))(e4)

# Bottleneck, intermediario entre el encoder y el decoder

b = layers.Conv2D(256, (3, 3), padding='same')(e4)
b = layers.BatchNormalization()(b)
b = layers.Activation('relu')(b)
b = layers.Conv2D(256, (3, 3), padding='same')(b)
b = layers.BatchNormalization()(b)
b = layers.Activation('relu')(b)


# ahora pasamos a la ultima parte de la arquitectura, el decoder, el cual recupera las dimensiones de la imagen orginal y junto con las skip connections genera el mapa final de caracteristicas semanticas, con clases

# Decoder - nivel 4

d4 = layers.Conv2DTranspose(128, (2, 2), strides=2, padding='same')(b)
d4 = layers.Concatenate()([d4, skip_connection4])   # concatenamos con la skip conection del mismo nivel en el encoder, esto hace que las features se dupliquen, por lo que haremos convolucion para mezclarlas
d4 = layers.Conv2D(128, (3, 3), padding='same')(d4)
d4 = layers.BatchNormalization()(d4)
d4 = layers.Activation('relu')(d4)
d4 = layers.Conv2D(128, (3, 3), padding='same')(d4) # la segunda convolucion es para generar las features del nivel
d4 = layers.BatchNormalization()(d4)
d4 = layers.Activation('relu')(d4)

# Decoder - nivel 3

d3 = layers.Conv2DTranspose(64, (2, 2), strides=2, padding='same')(d4)
d3 = layers.Concatenate()([d3, skip_connection3])
d3 = layers.Conv2D(64, (3, 3), padding='same')(d3)
d3 = layers.BatchNormalization()(d3)
d3 = layers.Activation('relu')(d3)
d3 = layers.Conv2D(64, (3, 3), padding='same')(d3)
d3 = layers.BatchNormalization()(d3)
d3 = layers.Activation('relu')(d3)

# Decoder - nivel 2

d2 = layers.Conv2DTranspose(32, (2,2), strides = 2, padding='same')(d3)
d2 = layers.Concatenate()([d2, skip_connection2])
d2 = layers.Conv2D(32, (3, 3), padding='same')(d2)
d2 = layers.BatchNormalization()(d2)
d2 = layers.Activation('relu')(d2)
d2 = layers.Conv2D(32, (3, 3), padding='same')(d2)
d2 = layers.BatchNormalization()(d2)
d2 = layers.Activation('relu')(d2)

# Decoder - nivel 1

d1 = layers.Conv2DTranspose(16, (2,2), strides = 2, padding='same')(d2)
d1 = layers.Concatenate()([d1, skip_connection1])
d1 = layers.Conv2D(16, (3, 3), padding='same')(d1)
d1 = layers.BatchNormalization()(d1)
d1 = layers.Activation('relu')(d1)
d1 = layers.Conv2D(16, (3, 3), padding='same')(d1)
d1 = layers.BatchNormalization()(d1)
d1 = layers.Activation('relu')(d1)

# Luego generamos la salida a través de una convolución con un kernel 1x1, la cual le asigna a cada pixel una clase, como son solo 2, en este caso usaremos sigmoide como activación, con un umbral de clasificacion tao

salida = layers.Conv2D(1, (1, 1), activation='sigmoid')(d1)

# Ahora generamos nuestro modelo
U_net = keras.Model(inputs=entrada, outputs=salida)
U_net.summary()

Ahora vamos a comprobar que es **la misma arquitectura de la v1**: exigimos la cifra exacta de parámetros, tanto con Keras como con la fórmula. Conteo analítico (conv: $9\,c_{in}f+f$; BN: $4f$, de los cuales $2f$ no entrenables; convT: $4\,c_{in}f+f$): encoder y cuello $=1{,}182{,}736$; decoder $=574{,}848+144{,}064+36{,}192+9{,}136$; salida $=17$; total $=1{,}946{,}993$, con $2{,}944$ no entrenables.

In [ ]:
def parametros_analiticos():  # conteo por fórmula, independiente de Keras
    def blq(cin, f):  # parámetros de un bloque base (conv + BN + conv + BN)
        return (9 * cin * f + f) + 4 * f + (9 * f * f + f) + 4 * f
    total, cin = 0, 3  # acumulado y canales de entrada
    for f in (16, 32, 64, 128):  # encoder
        total += blq(cin, f)
        cin = f
    total += blq(128, 256)  # cuello de botella
    cin = 256
    for f in (128, 64, 32, 16):  # decoder
        total += (4 * cin * f + f) + blq(2 * f, f)  # convT + bloque tras concatenar
        cin = f
    return total + (16 * 1 + 1)  # capa de salida 1x1


n_total = U_net.count_params()  # parámetros totales
n_train = int(sum(np.prod(w.shape) for w in U_net.trainable_weights))  # entrenables
n_no = int(sum(np.prod(w.shape) for w in U_net.non_trainable_weights))  # no entrenables
print(f'total={n_total:,} | entrenables={n_train:,} | no entrenables={n_no:,} | analítico={parametros_analiticos():,}')
assert n_total == PARAMS_ESPERADOS == parametros_analiticos()  # cifra exacta
assert n_train == 1944049 and n_no == 2944  # desglose exacto

Ahora vamos a definir la función de pérdida y las métricas, **iguales a las del 03**. Como el pez ocupa $\approx 15\,\%$ de los píxeles, combinamos la entropía cruzada binaria, que penaliza cada píxel, y la pérdida Dice, que mide el solapamiento y no se deja engañar por el desbalance.

Con $p_i$ la probabilidad predicha, $y_i$ la máscara real y $N$ los píxeles de una imagen:

$$\text{Dice} = \frac{2\sum_i p_i\, y_i + \varepsilon}{\sum_i p_i + \sum_i y_i + \varepsilon}, \qquad \mathcal{L} = \text{BCE} + \left(1 - \text{Dice}\right)$$

La métrica principal es el IoU por imagen, promediado sobre las imágenes (mIoU), con la predicción binarizada en $\tau = 0.5$:

$$\text{IoU} = \frac{|P \cap Y|}{|P \cup Y|}$$

In [ ]:
UMBRAL_MASCARA = 0.5                              # tau durante el entrenamiento; se ajusta sobre val en la sección 5
EJES = [1, 2, 3]                                  # alto, ancho y canal: una cifra por imagen

def dice_coef(y_true, y_pred, eps=1e-6):
    inter = tf.reduce_sum(y_true * y_pred, axis=EJES)                              # solapamiento suave
    suma = tf.reduce_sum(y_true, axis=EJES) + tf.reduce_sum(y_pred, axis=EJES)
    return tf.reduce_mean((2.0 * inter + eps) / (suma + eps))                      # promedio sobre el lote

def bce_dice(y_true, y_pred):
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))         # término por píxel
    return bce + (1.0 - dice_coef(y_true, y_pred))                                 # más el término de solapamiento

def iou_por_imagen(y_true, y_pred, eps=1e-6):
    p = tf.cast(y_pred > UMBRAL_MASCARA, tf.float32)                               # binarizamos con tau
    inter = tf.reduce_sum(y_true * p, axis=EJES)
    union = tf.reduce_sum(y_true, axis=EJES) + tf.reduce_sum(p, axis=EJES) - inter
    return tf.reduce_mean((inter + eps) / (union + eps))                           # mIoU del lote

Ahora vamos a compilar el modelo, con Adam (tasa de aprendizaje $10^{-3}$ por defecto), como en el 03.

In [ ]:
U_net.compile(optimizer='adam', loss=bce_dice, metrics=[iou_por_imagen])
print('métricas:', [m.name for m in U_net.metrics] or 'se crean en la primera época')

## 4. Entrenamiento

Máximo $40$ épocas, `ReduceLROnPlateau` (factor $0.5$, paciencia $3$, mínimo $10^{-5}$), `EarlyStopping` (paciencia $8$) y `ModelCheckpoint`, todos sobre `val_iou_por_imagen` (modo máx). Referencia de la v1: $\approx 48$ s por época con $4933$ imágenes y $\approx 162$ s la primera época (compilación); imprimimos el tiempo por época.

El paso es **idempotente**: si `unet_v2.keras`, el historial y la marca de entrenamiento completo existen y `REENTRENAR = False`, se cargan y no se entrena. Poner `REENTRENAR = True` **sobrescribe** el modelo y el historial.

Ahora vamos a decidir si entrenamos o cargamos.

In [ ]:
REENTRENAR = False  # True = entrenar de nuevo y sobrescribir; False = cargar si ya existe un entrenamiento completo
completo = os.path.exists(P_V2) and os.path.exists(P_HIST) and os.path.exists(P_MARCA)  # entrenamiento terminado
parcial = os.path.exists(P_V2) or os.path.exists(P_HIST)  # restos de un entrenamiento
ENTRENAR = REENTRENAR or not completo  # decisión
if REENTRENAR:
    print('REENTRENAR=True: se entrena desde cero y se sobrescriben modelo e historial')
elif completo:
    print('Entrenamiento completo encontrado: se cargará, no se entrena')
elif parcial:
    print('Hay restos de un entrenamiento incompleto (sin marca de fin): se entrena de nuevo')
else:
    print('No hay entrenamiento previo: se entrena')

Ahora vamos a entrenar (si corresponde). Quitamos antes la marca de fin, de modo que un entrenamiento interrumpido nunca se confunda con uno completo, y la reescribimos al terminar.

In [ ]:
class TiempoPorEpoca(keras.callbacks.Callback):  # imprime el tiempo de cada época
    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()  # inicio de la época

    def on_epoch_end(self, epoch, logs=None):
        print(f'   época {epoch + 1}: {time.time() - self.t0:.1f} s')  # duración


if ENTRENAR:
    if os.path.exists(P_MARCA):  # invalida la marca anterior
        os.remove(P_MARCA)
    callbacks = [
        keras.callbacks.ReduceLROnPlateau(monitor='val_iou_por_imagen', mode='max', factor=0.5, patience=3, min_lr=1e-5, verbose=1),  # baja lr
        keras.callbacks.EarlyStopping(monitor='val_iou_por_imagen', mode='max', patience=8, restore_best_weights=True, verbose=1),  # parada temprana
        keras.callbacks.ModelCheckpoint(P_V2, monitor='val_iou_por_imagen', mode='max', save_best_only=True, verbose=1),  # mejor modelo
        keras.callbacks.CSVLogger(P_HIST, append=False),  # historial
        TiempoPorEpoca(),  # tiempos
    ]
    t_ini = time.time()  # inicio del entrenamiento
    U_net.fit(ds_tr, validation_data=ds_va, epochs=EPOCAS_MAX, callbacks=callbacks, verbose=2)  # entrena
    t_total = time.time() - t_ini  # duración total
    h = pd.read_csv(P_HIST)  # historial escrito por CSVLogger
    mejor = int(h['val_iou_por_imagen'].idxmax())  # índice de la mejor época
    escribir_json({'epocas_corridas': int(len(h)), 'mejor_epoca': mejor + 1,
                   'mejor_val_iou_por_imagen': float(h['val_iou_por_imagen'].iloc[mejor]),
                   'segundos_totales': round(t_total, 1),
                   'fecha': datetime.datetime.now().isoformat(timespec='seconds')}, P_MARCA)  # marca de fin
    print(f'Entrenamiento terminado en {t_total / 60:.1f} min')
else:
    print('Entrenamiento omitido (se carga el existente)')

Ahora vamos a cargar desde disco el **mejor** modelo (el del `ModelCheckpoint`, que coincide con el criterio de `restore_best_weights`) y a verificar de nuevo la cifra de parámetros. Así el resto del cuaderno usa siempre el mismo objeto, se haya entrenado o no en esta sesión.

In [ ]:
modelo_v2 = keras.models.load_model(P_V2, compile=False)  # mejor v2 (solo arquitectura y pesos)
assert modelo_v2.count_params() == PARAMS_ESPERADOS  # misma arquitectura que la v1
hist = pd.read_csv(P_HIST)  # historial
with open(P_MARCA, encoding='utf-8') as f:  # marca de entrenamiento
    marca = json.load(f)
print('Resumen del entrenamiento:', marca)
print(hist.tail(3).to_string(index=False))

## 5. Curvas, umbral $\tau$ y guardado

Ahora vamos a graficar las curvas de pérdida y de `iou_por_imagen` (entrenamiento y validación), marcando la mejor época.

In [ ]:
ep = hist['epoch'] + 1  # épocas desde 1
mejor = int(hist['val_iou_por_imagen'].idxmax())  # índice de la mejor época
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))  # dos paneles
ejes[0].plot(ep, hist['loss'], label='entrenamiento')  # pérdida de train
ejes[0].plot(ep, hist['val_loss'], label='validación')  # pérdida de val
ejes[0].set_xlabel('época')
ejes[0].set_ylabel('BCE + (1 - Dice)')
ejes[0].legend()
ejes[1].plot(ep, hist['iou_por_imagen'], label='entrenamiento (aumentado)')  # IoU de train
ejes[1].plot(ep, hist['val_iou_por_imagen'], label='validación')  # IoU de val
ejes[1].axvline(mejor + 1, color='gray', linestyle='--', label=f'mejor época = {mejor + 1}')  # mejor época
ejes[1].set_xlabel('época')
ejes[1].set_ylabel('IoU por imagen (umbral 0.5, contra SAM 2)')
ejes[1].legend()
fig.suptitle('U-Net v2 (etiquetas SAM 2): curvas de entrenamiento')
guardar_figura(fig, f'{DIR_FIGS}/{ETIQUETAS}_unet_v2_curvas.png')  # guarda
plt.show()

Ahora vamos a elegir el umbral. Barremos $\tau\in\{0.10,0.15,\dots,0.90\}$ (la misma rejilla del 03) sobre val (solo `sam2_qc_ok`) y tomamos el que maximiza

$$\mathrm{mIoU}(\tau)=\frac{1}{N}\sum_{i=1}^{N}\frac{|\hat y_i^{\tau}\cap y_i|+\varepsilon}{|\hat y_i^{\tau}\cup y_i|+\varepsilon},\qquad \hat y_i^{\tau}=\mathbb{1}[p_i>\tau].$$

Si hay empate se toma el menor $\tau$. Ese mIoU es **contra pseudo-máscaras de SAM 2** y **no** es comparable con el $0.8334$ de la v1 (contra GrabCut).

In [ ]:
P_va = modelo_v2.predict(ds_va, verbose=0)[..., 0]  # probabilidades en val, mismo orden que X_va
GT_va = M_va[..., 0] > 127  # máscaras SAM 2 de val como booleanos
assert P_va.shape == GT_va.shape  # alineación por posición del mismo arreglo filtrado
mious = []  # mIoU por umbral
for tau in TAUS:  # barrido
    B = P_va > tau  # predicción binaria
    inter = np.logical_and(B, GT_va).sum(axis=(1, 2)).astype(np.float64)  # intersección por imagen
    union = np.logical_or(B, GT_va).sum(axis=(1, 2)).astype(np.float64)  # unión por imagen
    mious.append(float(np.mean((inter + EPS) / (union + EPS))))  # IoU medio por imagen
mious = np.array(mious)
print('tau   mIoU(val, vs SAM 2)')
for t, m in zip(TAUS, mious):  # tabla
    print(f'{t:.2f}   {m:.4f}')
i_best = int(np.argmax(mious))  # mejor umbral (el menor en caso de empate)
TAU_V2, MIOU_V2 = float(TAUS[i_best]), float(mious[i_best])  # umbral y mIoU
print('=' * 72)
print(f'tau elegido = {TAU_V2:.2f} | mIoU por imagen en val = {MIOU_V2:.4f}')
print('ESTE mIoU ES CONTRA PSEUDO-MASCARAS DE SAM 2 Y NO ES COMPARABLE CON EL 0.8334 DE LA v1 (CONTRA GRABCUT).')
print('=' * 72)
print(f'Control mecánico: mIoU en tau=0.5 = {mious[list(TAUS).index(0.5)]:.4f} | val_iou_por_imagen de la mejor época = {hist["val_iou_por_imagen"].iloc[mejor]:.4f}')
escribir_json({'tau': TAU_V2, 'miou_val': round(MIOU_V2, 4)}, P_V2_TAU)  # guarda el umbral
fig, ax = plt.subplots(figsize=(6, 4))  # figura del barrido
ax.plot(TAUS, mious, marker='o')  # curva mIoU(tau)
ax.axvline(TAU_V2, color='gray', linestyle='--', label=f'tau elegido = {TAU_V2:.2f}')  # elegido
ax.set_xlabel('umbral tau')
ax.set_ylabel('mIoU por imagen en val (contra SAM 2)')
ax.set_title('U-Net v2: selección de tau')
ax.legend()
guardar_figura(fig, f'{DIR_FIGS}/{ETIQUETAS}_umbral_v2.png')  # guarda
plt.show()

## 6. Comparación visual v1 contra v2

Comparamos **a ojo** sobre las $30$ imágenes de `muestra_val.csv` (estratos `aleatoria` $=15$, `peor` $=8$, `descartada` $=7$). Para cada imagen predecimos con la v1 y con la v2 igual: entrada RGB $256\times256$ con `INTER_AREA` y $/255$, probabilidades ampliadas a $500\times500$ con interpolación bilineal y luego el $\tau$ de cada modelo; dibujamos el contorno de mayor área sobre la imagen original. El id $1..30$ sigue el **orden del CSV**.

**Pre-registro.** Antes de mirar las cuadrículas fijamos `REGLA_VENTAJA_AZAR`: el número mínimo de imágenes `bien` adicionales de la v2 sobre la v1 en el estrato `aleatoria` (de $15$) que bastaría para adoptar la v2. Debe ser un entero entre $1$ y $15$ y se decide por criterio propio, no después de ver los resultados. Con solo $15$ imágenes el ruido de una calificación visual es grande: diferencias de una o dos imágenes pueden ser puro azar de la calificación.

Ahora vamos a fijar la regla. La celda siguiente **falla a propósito** si queda en `None`.

In [ ]:
REGLA_VENTAJA_AZAR = 5  # fijada por Jairo el 2026-11-19 ANTES de ver las cuadrículas (entero 1..15)
assert isinstance(REGLA_VENTAJA_AZAR, int) and not isinstance(REGLA_VENTAJA_AZAR, bool), 'Fija REGLA_VENTAJA_AZAR (entero) antes de mirar las cuadrículas'  # pre-registro
assert 1 <= REGLA_VENTAJA_AZAR <= N_ESTRATO['aleatoria'], 'La regla debe estar entre 1 y 15'  # rango válido
print('Regla fijada antes de mirar las cuadrículas: la v2 necesita al menos', REGLA_VENTAJA_AZAR, 'imágenes bien más que la v1 en aleatoria')

Ahora vamos a leer `muestra_val.csv`, asignar los ids $1..30$ en el orden del CSV y comprobar que las $30$ imágenes son de val y que los estratos tienen los tamaños esperados.

In [ ]:
muestra = pd.read_csv(P_MUESTRA)  # muestra de val (cuaderno A)
assert {'filepath', 'estrato'} <= set(muestra.columns)  # columnas del contrato
muestra['id'] = np.arange(1, len(muestra) + 1)  # id 1..30 en el orden del CSV
assert len(muestra) == 30 and muestra['filepath'].is_unique  # 30 imágenes distintas
assert muestra['estrato'].value_counts().to_dict() == N_ESTRATO, muestra['estrato'].value_counts().to_dict()  # 15/8/7
assert set(muestra['filepath']) <= set(splits.loc[splits['split'] == 'val', 'filepath'])  # todas son de val
ESTRATO_DE_ID = dict(zip(muestra['id'], muestra['estrato']))  # id -> estrato
IDS_POR_ESTRATO = {e: muestra.loc[muestra['estrato'] == e, 'id'].tolist() for e in ESTRATOS}  # estrato -> ids
muestra['stem'] = muestra['filepath'].map(stem)  # stem de cada imagen
print(muestra[['id', 'estrato', 'stem']].to_string(index=False))

Ahora vamos a cargar las imágenes originales (RGB, $500\times500$), los dos modelos (con `compile=False`, solo lectura) con sus umbrales, y las máscaras de GrabCut desde `masks.zip` emparejadas por *stem*.

In [ ]:
imgs = []  # imágenes originales RGB 500x500
for fp in muestra['filepath']:  # en el orden del CSV
    bgr = cv2.imread(ruta_img(fp))  # cv2 lee BGR
    assert bgr is not None, f'no se pudo leer {fp}'
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)  # a RGB
    assert rgb.shape == (500, 500, 3), f'tamaño inesperado en {fp}: {rgb.shape}'  # contrato 500x500
    imgs.append(rgb)

modelo_v1 = keras.models.load_model(P_V1, compile=False)  # v1, solo lectura
with open(P_V1_TAU, encoding='utf-8') as f:  # umbral de la v1
    TAU_V1 = float(json.load(f)['tau'])
with open(P_V2_TAU, encoding='utf-8') as f:  # umbral de la v2 (recién guardado)
    TAU_V2 = float(json.load(f)['tau'])
print(f'v1: {modelo_v1.count_params():,} parámetros, tau={TAU_V1} | v2: {modelo_v2.count_params():,} parámetros, tau={TAU_V2}')

with zipfile.ZipFile(P_ZIP_GRABCUT) as zf:  # masks.zip (GrabCut)
    zg = {stem(n): n for n in zf.namelist() if n.lower().endswith('.png') and '__MACOSX' not in n and not os.path.basename(n).startswith('.')}  # stem -> ruta interna
    gc = []  # máscaras GrabCut de la muestra
    for s in muestra['stem']:  # en el orden del CSV
        if s not in zg:  # sin máscara GrabCut
            print('AVISO: sin máscara GrabCut para', s)
            gc.append(None)
            continue
        m = cv2.imdecode(np.frombuffer(zf.read(zg[s]), np.uint8), cv2.IMREAD_GRAYSCALE)  # decodifica
        if m.shape != (500, 500):  # por si acaso
            m = cv2.resize(m, (500, 500), interpolation=cv2.INTER_NEAREST)
        gc.append(m > 127)  # booleana

Ahora vamos a predecir con ambos modelos y a extraer el contorno de mayor área. La entrada sigue el contrato de la U-Net (`INTER_AREA`, $/255$); las probabilidades se amplían a $500\times500$ con `INTER_LINEAR` y se binarizan con el $\tau$ de cada modelo.

In [ ]:
X30 = np.stack([cv2.resize(i, (IMG, IMG), interpolation=cv2.INTER_AREA) for i in imgs]).astype('float32') / 255.0  # (30,256,256,3)


def contorno_mayor(binaria):  # contorno externo de mayor área
    cont, _ = cv2.findContours(binaria.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # contornos
    if len(cont) == 0:  # sin región
        return None
    return max(cont, key=cv2.contourArea)  # el de mayor área


def contornos_modelo(modelo, tau):  # contorno de cada imagen según un modelo y su umbral
    P = modelo.predict(X30, batch_size=BATCH, verbose=0)[..., 0]  # probabilidades 256x256
    res = []  # contornos
    for p in P:  # por imagen
        p500 = cv2.resize(p, (500, 500), interpolation=cv2.INTER_LINEAR)  # amplía con bilineal
        res.append(contorno_mayor(p500 > tau))  # binariza con tau y extrae contorno
    return res


CONT = {'GrabCut': [contorno_mayor(m) if m is not None else None for m in gc],  # contornos GrabCut
        'v1': contornos_modelo(modelo_v1, TAU_V1),  # contornos v1
        'v2': contornos_modelo(modelo_v2, TAU_V2)}  # contornos v2
for k, v in CONT.items():  # control: imágenes sin contorno
    print(k, '- imágenes sin contorno:', sum(c is None for c in v))

Ahora vamos a dibujar las cuadrículas, agrupadas por estrato y con el id $1..30$ visible. Columnas: original | GrabCut | v1 | v2. Calificamos mirando estas figuras, en la sección 7.

In [ ]:
COLORES = {'GrabCut': (255, 200, 0), 'v1': (0, 255, 255), 'v2': (255, 0, 0)}  # colores RGB de cada contorno
COLUMNAS = ['original', 'GrabCut', 'v1', 'v2']  # columnas de la cuadrícula
POR_FIGURA = 5  # filas por figura

for estrato in ESTRATOS:  # un grupo por estrato
    ids = IDS_POR_ESTRATO[estrato]  # ids del estrato en el orden del CSV
    for pag, ini in enumerate(range(0, len(ids), POR_FIGURA), start=1):  # páginas de 5 filas
        trozo = ids[ini:ini + POR_FIGURA]  # ids de esta figura
        fig, ejes = plt.subplots(len(trozo), 4, figsize=(14, 3.6 * len(trozo)), squeeze=False)  # cuadrícula
        for r, i in enumerate(trozo):  # cada fila = una imagen
            k = i - 1  # posición en las listas (id = posición + 1 por construcción del CSV)
            for c, col in enumerate(COLUMNAS):  # cada columna
                img = imgs[k].copy()  # copia para dibujar
                if col != 'original' and CONT[col][k] is not None:  # dibuja el contorno
                    cv2.drawContours(img, [CONT[col][k]], -1, COLORES[col], 2)
                ax = ejes[r, c]
                ax.imshow(img)
                ax.axis('off')
                if r == 0:
                    ax.set_title(col, fontsize=12)  # nombre de columna
                if c == 0:  # id grande sobre la original
                    ax.text(8, 40, f'{i}', color='white', fontsize=22, weight='bold', bbox=dict(facecolor='black', alpha=0.7))
                    ax.text(8, 485, muestra['stem'].iloc[k], color='white', fontsize=9, bbox=dict(facecolor='black', alpha=0.6))
        fig.suptitle(f'Estrato {estrato} (ids {trozo[0]} a {trozo[-1]}) - página {pag}', y=1.0)  # título
        plt.tight_layout()
        guardar_figura(fig, f'{DIR_FIGS}/{ETIQUETAS}_comparacion_{estrato}_p{pag}.png')  # guarda
        plt.show()

## 7. Calificación visual y regla de decisión

Ahora vamos a calificar. Mirando las cuadrículas, asignamos **cada id** $1..30$ a exactamente una lista por modelo: `bien`, `parcial` o `mal`. La celda valida que no falte ni sobre ninguno y que no haya repetidos.

Conteo y decisión **por estrato**, sin mezclarlos en un solo promedio. Con $b_m^{e}$ las imágenes `bien` del modelo $m$ en el estrato $e$, se adopta la v2 solo si

$$b_{v2}^{\text{aleatoria}}-b_{v1}^{\text{aleatoria}}\ \ge\ \texttt{REGLA\_VENTAJA\_AZAR}\quad\text{y}\quad b_{v2}^{\text{peor}}\ge b_{v1}^{\text{peor}}\quad\text{y}\quad b_{v2}^{\text{descartada}}\ge b_{v1}^{\text{descartada}}.$$

En cualquier otro caso se conserva la v1. Si hay empate o la diferencia es menor que la regla, el resultado es "empate: se conserva la v1 por defecto, ya evaluada y documentada". Recordatorio: con $15$ imágenes el ruido es grande.

Si la v2 resulta **claramente peor** (por ejemplo si pasa a `mal`), antes de concluir algo sobre las etiquetas descartamos las causas **mecánicas**: máscaras desalineadas respecto a las imágenes, caché equivocada (orden de `filepath`), falta de la división entre $255$, y $\tau$ sin reelegir para la v2.

La celda falla mientras `CALIF` esté vacío; es intencional.

In [ ]:
CALIF = {  # <- COMPLETAR con los ids 1..30 (cada id en exactamente una lista por modelo)
    'v1': {'bien': [], 'parcial': [], 'mal': []},
    'v2': {'bien': [], 'parcial': [], 'mal': []},
}
CATS = ('bien', 'parcial', 'mal')  # categorías de calificación
TODOS = set(range(1, 31))  # ids válidos

assert isinstance(REGLA_VENTAJA_AZAR, int) and 1 <= REGLA_VENTAJA_AZAR <= 15, 'REGLA_VENTAJA_AZAR no está fijada (sección 6)'  # regla fijada
for mod in ('v1', 'v2'):  # validación por modelo
    vistos = [i for c in CATS for i in CALIF[mod][c]]  # todos los ids asignados
    dup = sorted({i for i in vistos if vistos.count(i) > 1})  # repetidos
    faltan = sorted(TODOS - set(vistos))  # sin asignar
    sobran = sorted(set(vistos) - TODOS)  # fuera de rango
    assert not dup, f'{mod}: ids en más de una lista: {dup}'
    assert not faltan, f'{mod}: ids sin calificar: {faltan}'
    assert not sobran, f'{mod}: ids fuera de 1..30: {sobran}'

cuenta = {mod: {e: {c: sum(ESTRATO_DE_ID[i] == e for i in CALIF[mod][c]) for c in CATS} for e in ESTRATOS} for mod in ('v1', 'v2')}  # conteos
for e in ESTRATOS:  # los conteos de cada estrato deben sumar su tamaño
    for mod in ('v1', 'v2'):
        assert sum(cuenta[mod][e].values()) == N_ESTRATO[e]
tabla = pd.DataFrame({(mod, c): {e: cuenta[mod][e][c] for e in ESTRATOS} for mod in ('v1', 'v2') for c in CATS})  # tabla por estrato
print(tabla.to_string())  # sin promedio global

ventaja = cuenta['v2']['aleatoria']['bien'] - cuenta['v1']['aleatoria']['bien']  # ventaja de v2 en aleatoria
c_ventaja = ventaja >= REGLA_VENTAJA_AZAR  # condición 1
c_peor = cuenta['v2']['peor']['bien'] >= cuenta['v1']['peor']['bien']  # condición 2
c_desc = cuenta['v2']['descartada']['bien'] >= cuenta['v1']['descartada']['bien']  # condición 3
decision = 'v2' if (c_ventaja and c_peor and c_desc) else 'v1'  # regla de decisión
if decision == 'v2':
    motivo = f'se adopta la v2: ventaja {ventaja} >= regla {REGLA_VENTAJA_AZAR} en aleatoria y sin pérdida de bien en peor ni en descartada'
elif c_peor and c_desc and 0 <= ventaja < REGLA_VENTAJA_AZAR:
    motivo = 'empate: se conserva la v1 por defecto, ya evaluada y documentada'
else:
    razones = []  # causas de no adoptar
    if not c_ventaja:
        razones.append(f'ventaja en aleatoria = {ventaja} < regla {REGLA_VENTAJA_AZAR}')
    if not c_peor:
        razones.append('la v2 tiene menos bien que la v1 en peor')
    if not c_desc:
        razones.append('la v2 tiene menos bien que la v1 en descartada')
    motivo = 'se conserva la v1: ' + '; '.join(razones)
print('=' * 72)
print('DECISIÓN:', decision.upper(), '|', motivo)
print('=' * 72)
if ventaja <= -REGLA_VENTAJA_AZAR:  # v2 claramente peor
    print('La v2 es claramente peor en aleatoria. Descartar primero causas mecánicas:')
    print(' 1) máscaras desalineadas (revisar las cuadrículas de la sección 1)')
    print(' 2) caché equivocada (orden de filepath, split correcto)')
    print(' 3) falta de la división entre 255 en alguna entrada')
    print(' 4) tau sin reelegir para la v2 (sección 5)')

escribir_json({  # registro de la decisión
    'fecha': datetime.datetime.now().isoformat(timespec='seconds'),
    'regla_ventaja_azar': REGLA_VENTAJA_AZAR,
    'tau_v1': TAU_V1, 'tau_v2': TAU_V2,
    'conteos_por_estrato': cuenta,
    'calificacion_ids': CALIF,
    'ventaja_bien_aleatoria': ventaja,
    'condiciones': {'ventaja_aleatoria': bool(c_ventaja), 'peor_sin_perdida': bool(c_peor), 'descartada_sin_perdida': bool(c_desc)},
    'decision': decision,
    'motivo': motivo,
    'nota': 'Calificación visual (a ojo), estratos n=15/8/7 sin mezclar; con n=15 el ruido es grande. No hay IoU contra anotaciones.'},
    P_SELECCION)  # escritura atómica
print('Guardado:', P_SELECCION)

## 8. Test (opcional y protegido)

El test se usa **una sola vez** y **solo si se adopta la v2 y nosotros lo pedimos**. Esta sección no corre mientras `CONFIRMO_TEST = False`. Si lo ponemos en `True`: (1) se exige que `sam2_seleccion.json` diga `decision = v2`; (2) se escribe `sam2_unet_v2_test_registro.json` con fecha, modelo y criterio **antes** de evaluar; (3) si ese registro ya existe, el cuaderno **se niega** a evaluar otra vez. Borrar el registro es una decisión consciente que habría que documentar.

El mIoU resultante es contra pseudo-máscaras de SAM 2 y **no es comparable** con el $0.8344$ de la v1 en test (contra GrabCut).

Ahora vamos a escribir el registro (solo si lo confirmamos).

In [ ]:
CONFIRMO_TEST = False  # True solo si se adoptó la v2 y queremos usar el test UNA vez
TEST_AUTORIZADO = False  # lo activa esta misma celda si todo está en regla
if not CONFIRMO_TEST:
    print('Sección de test omitida (CONFIRMO_TEST = False)')
else:
    if os.path.exists(P_REG_TEST):  # el test ya se usó
        raise RuntimeError(f'El test ya fue evaluado (existe {P_REG_TEST}). No se evalúa de nuevo.')
    with open(P_SELECCION, encoding='utf-8') as f:  # decisión de la sección 7
        sel = json.load(f)
    if sel.get('decision') != 'v2':  # solo se evalúa si se adoptó la v2
        raise RuntimeError('La decisión de la sección 7 no es v2: el test no corresponde.')
    escribir_json({
        'fecha': datetime.datetime.now().isoformat(timespec='seconds'),
        'modelo': 'models/unet_v2.keras',
        'md5_modelo': md5_archivo(P_V2),
        'tau': TAU_V2,
        'etiquetas': ETIQUETAS,
        'criterio': 'v2 adoptada según la regla de sam2_seleccion.json (calificación visual por estratos); test usado una sola vez',
        'regla_ventaja_azar': sel.get('regla_ventaja_azar')}, P_REG_TEST)  # registro ANTES de evaluar
    TEST_AUTORIZADO = True
    print('Registro escrito; el test se evalúa una única vez en la celda siguiente')

Ahora vamos a evaluar la v2 con su $\tau$ sobre el test filtrado por `sam2_qc_ok` y a reportar el mIoU por imagen contra las máscaras de SAM 2.

In [ ]:
if not TEST_AUTORIZADO:
    print('Evaluación de test omitida')
else:
    X_te, M_te, fp_te = cargar_split('test')  # test filtrado por sam2_qc_ok
    ds_te = crear_dataset(X_te, M_te, False)  # sin aumento y sin mezcla
    P_te = modelo_v2.predict(ds_te, verbose=0)[..., 0]  # probabilidades
    GT_te = M_te[..., 0] > 127  # máscaras SAM 2
    B = P_te > TAU_V2  # binariza con el tau de la v2
    inter = np.logical_and(B, GT_te).sum(axis=(1, 2)).astype(np.float64)  # intersección
    union = np.logical_or(B, GT_te).sum(axis=(1, 2)).astype(np.float64)  # unión
    miou_te = float(np.mean((inter + EPS) / (union + EPS)))  # IoU medio por imagen
    print('=' * 72)
    print(f'mIoU por imagen en test (v2, tau={TAU_V2:.1f}, n={len(fp_te)}) = {miou_te:.4f}')
    print('ES CONTRA PSEUDO-MASCARAS DE SAM 2 Y NO ES COMPARABLE CON EL 0.8344 DE LA v1 (CONTRA GRABCUT).')
    print('=' * 72)
    escribir_json({'fecha': datetime.datetime.now().isoformat(timespec='seconds'),
                   'modelo': 'models/unet_v2.keras', 'tau': TAU_V2,
                   'n_test_sam2_qc_ok': int(len(fp_te)), 'miou_test_vs_sam2': round(miou_te, 4),
                   'nota': 'Contra pseudo-máscaras de SAM 2; no comparable con el 0.8344 de la v1 (contra GrabCut).'}, P_RES_TEST)
    print('Guardado:', P_RES_TEST)

## Cierre

Ahora tenemos la U-Net v2 reentrenada solo con etiquetas distintas, su umbral, las curvas, las cuadrículas v1 contra v2 y la decisión registrada por estrato. Recordatorio final: la calificación es visual (a ojo), con $n=15/8/7$; la v1 sigue siendo la referencia salvo que la regla de la sección 7 diga lo contrario.